# **Libraries**

In [ ]:
!pip install textblob
!pip install gensim
!pip install numpy
!pip install matplotlib
!pip install pandas
!pip install spacy
!pip install nltk
!pip install scikit-learn
!pip install seaborn

In [ ]:
# Basic Libraries
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import csv

import spacy
# Load the spaCy English model
nlp = spacy.load('en_core_web_sm')

import string
from collections import Counter

# NLP Libraries - Data Preprocessing
import nltk
from nltk.corpus import stopwords, words
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('words')
nltk.download('wordnet')
nltk.download('vader_lexicon')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')

import re

# Dictionaries
from collections import defaultdict

# Save stop words to a variable
stop_words = stopwords.words()

# NLTK Corpus
corpus = set(nltk.corpus.words.words())

# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Topic Modelling
# Gensim
import gensim, spacy, logging, warnings
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel
from gensim.models.ldamodel import LdaModel
from gensim.models.nmf import Nmf
from gensim.models import LsiModel

# **Load Dataset**

In [ ]:
# Read the CSV file
data = pd.read_csv('phishing_ceas_08.csv')

# Display the contents of the CSV file
print(data.head())

In [ ]:
# Define a custom function to combine columns
def combine_columns(row):
    return str(row['subject']) + ' ' + row['body']

# Apply the custom function to create a new column 'Combined'
data['body'] = data.apply(combine_columns, axis=1)

print(data.head())

In [ ]:
# Create a new dataframe and save the body of each email
df_emails = pd.DataFrame()
df_emails[['Whole_Email', 'Url_Exists']] = data[['body', 'urls']]
df_emails['Index'] = df_emails.index
df_emails = df_emails.iloc[:,[2,0,1]]

In [ ]:
NUM_OF_EMAILS = 17312

# Print total number of phishing emails
print("Total number of phishing emails:", len(df_emails))

df_emails = df_emails[:NUM_OF_EMAILS]
print("New total number of phishing emails:", len(df_emails))

# **Data Preprocessing & Features**

In [ ]:
def find_email_length(email):
  '''
  Get the length of each email.

  Args:
    email (str): The body (text) of each email

  Return:
    word_count (int): Number of words / Length of the email
  '''

  # Split the email into words and count them to get its length
  word_count = len(word_tokenize(email))

  return word_count

df_emails['Length'] = df_emails['Whole_Email'].apply(find_email_length)

print(df_emails)

In [ ]:
# Save the column to a variable
l = df_emails['Length']

# Features of emails
print("Total number of words:", np.sum(l))
print("Maximum Length:", np.max(l))
print("Minimum Length:", np.min(l))
print("Anerage Length:", np.mean(l))

* **Uppercase Letters Distribution**
1. Per Email


In [ ]:
def calculate_uppercase_percentage(email):
  '''
  Calculate the percentage of uppercase words per email.

  Args:
    email (str): The body (text) of each email

  Return:
    If there are uppercase words, return their percentage else 0
  '''

  # Split the email into words
  words = word_tokenize(email)

  # Get email's length
  length = len(words)

  # Count uppercase words in an email
  uppercase_words = sum(1 for word in words if word.isupper())

  return (uppercase_words / length) * 100 if length > 0 else 0

# Analyze the per email distribution of uppercase letters
df_emails['Per_Email_Uppercase_Percentage'] = df_emails['Whole_Email'].apply(calculate_uppercase_percentage)
print(df_emails)

In [ ]:
# Boxplot of per email distribution
plt.figure(figsize=(10, 6))
plt.boxplot(df_emails['Per_Email_Uppercase_Percentage'])
plt.title('Per Email Distribution of Uppercase Words')
plt.ylabel('Percentage of Uppercase Words')
plt.suptitle('Phishing Emails', y=1, fontsize=14)
plt.show()

In [ ]:
def get_boxplot_fences(column):
  '''
  Get the upper ("maximum") and lower("minimum") fences of the boxplot.

  Args:
    column (dataframe column): The column used for the boxplot.

  Return:
    The values of the fences.
  '''

  quantile_1 = column.quantile(0.25)
  quantile_3 = column.quantile(0.75)
  iqr = quantile_3 - quantile_1

  lower_fence = quantile_1 - (1.5 * iqr)
  upper_fence = quantile_3 + (1.5 * iqr)

  return lower_fence, upper_fence

lower_fence, upper_fence = get_boxplot_fences(df_emails['Per_Email_Uppercase_Percentage'])

# The emails that are above the upper fence -> outliers
outliers = df_emails.loc[df_emails['Per_Email_Uppercase_Percentage'] > upper_fence]
print(outliers)

2. Overall

In [ ]:
# Overall Distribution of Uppercase Letters
emails_list = df_emails['Whole_Email'].tolist()
emails_str = ''.join(emails_list)
overall_upper_perc = calculate_uppercase_percentage(emails_str)
print("Overall Distribution of Uppercase Letters:", overall_upper_perc)

* **URL Removal**

In [ ]:
def collect_and_remove_urls(email):
  '''
  Collect the urls and remove them from the text.

  Args:
    email (str): The body (text) of each email

  Return:
    The urls' list and the updated text without urls inside.
  '''
  URL_FORMAT = r'(?:https?://)?(?:www\.)?[a-zA-Z0-9-]+(?:\.[a-zA-Z]{2,})+(?:/[^\s]*)?'

  # Extract URLs using regex
  urls = re.findall(URL_FORMAT, email)

  # Remove URLs from the text
  text_without_urls = re.sub(URL_FORMAT, '', email)

  return urls, text_without_urls

df_emails[['Urls', 'Cleaned_Email']] = df_emails['Whole_Email'].apply(collect_and_remove_urls).apply(pd.Series)

print(df_emails)

* **Remove special characters**

In [ ]:
# Create an empty global list for the elements that will be removed

SEQ_OF_SPECIAL_CHARS = r'[^a-zA-Z0-9\s]{2,}'
SPECIAL_CHARS_FORMAT = r'[^a-zA-Z0-9\s]'

def remove_special_characters(email):
  '''
  Remove special chracters from the email, but save them in a list.

  Args:
    email (str): The body (text) of each email

  Return:
    The updated text without special characters.
  '''

  # Find all sequences of multiple special characters (>2 chars)
  special_chars = re.findall(SEQ_OF_SPECIAL_CHARS, email)

  # Keep only alphanumeric characters and spaces
  text_without_special_chars =  re.sub(SPECIAL_CHARS_FORMAT, ' ', email)

  return text_without_special_chars, special_chars

df_emails[['Cleaned_Email', 'Special_Characters']] = df_emails['Cleaned_Email'].apply(remove_special_characters).apply(pd.Series)


print(df_emails)

* **NER**



In [ ]:
def apply_ner(email):
  '''
  Apply Named Entity Recognition to the body of the email.

  Args:
    email (str): The body (text) of each email

  Return:
    A dictionary having as keys the
    words and as values their entities.
  '''

  # Process the text with SpaCy
  doc = nlp(email)

  # Extract named entities
  entities = [(ent.text, ent.label_) for ent in doc.ents]

  return entities

df_emails['NER_Clean_Results'] = df_emails['Cleaned_Email'].apply(apply_ner)

In [ ]:
# Create an empty list to save the recognized entities of each email
entities = []
for result in df_emails['NER_Clean_Results']:
  entities.append([tag[1] for tag in result])

# Flatten the list of lists which contains the entities of each email separately
all_entities = [tag for element in entities for tag in element]

# Count the occurencies of each tag
all_entities_counts = Counter(all_entities)

# Sort POS tags based on their frequency
sorted_all_entities_counts = dict(sorted(all_entities_counts.items(), key=lambda item: item[1], reverse=True))

# Use multiple colors
colors = sns.color_palette('colorblind', len(all_entities_counts))

# Create a barplot
plt.figure(figsize=(7, 6))
plt.bar(sorted_all_entities_counts.keys(), sorted_all_entities_counts.values(), color=colors)
plt.xticks(rotation=45, ha="right")
plt.xlabel('Recognized Entities')
plt.ylabel('Frequency')
plt.title('Recognized Entities Distribution')
plt.suptitle('Phishing Emails', y=1, fontsize=15)

*   **Spot the numbers and keep the previous word and the following word**




In [ ]:
def find_phrases_with_numbers(email):

  words = email.split()
  results = []

  for i in range(len(words)):
    if words[i].isdigit():
      prev_word = words[i-1] if i > 0 else None
      next_word = words[i+1] if i < len(words)-1 else None
      results.append((prev_word, words[i], next_word))

  return results


df_emails['Numbers_With_Context'] = df_emails['Cleaned_Email'].apply(find_phrases_with_numbers)

print(df_emails)

* **Remove stopwords**

In [ ]:
# Remove stopwords
def remove_stopwords(email):
  '''
  Remove all the stop words from the email.

  Args:
    email (str): The body (text) of each email

  Return:
    The updated text without stop words.
  '''

  # Split the email into words
  words_list = word_tokenize(email)

  # Use list comprehension to remove the stopwords
  email_without_stopwords = [ word for word in words_list if word.lower() not in stop_words ]

  return ' '.join(email_without_stopwords)

df_emails['Cleaned_Email'] = df_emails['Cleaned_Email'].apply(remove_stopwords)

print(df_emails)

In [ ]:
# Text after removing stopwords, but without converting it to lowercase
df_emails['Email_With_Original_Letters'] = df_emails['Cleaned_Email']

In [ ]:
# Convert to lowercase
def convert_to_lowercase(email):
  '''
  Convert text to lowercase.

  Args:
    email (str): The body (text) of each email

  Return:
    The same email but with lowercase letters.
  '''

  return email.lower()

df_emails['Cleaned_Email'] = df_emails['Cleaned_Email'].apply(convert_to_lowercase)

print(df_emails)

In [ ]:
def lemmatization(email):
  '''
  Lemmatize the email's text.

  Args:
    email (str): The body (text) of each email

  Return:
    The updated email body after grouping together different
    inflected forms of the same word
  '''
  # Process the text using spaCy
  doc = nlp(email)

  # Extract lemmatized tokens
  lemmatized_tokens = [token.lemma_ for token in doc]

  # Join the lemmatized tokens into a sentence
  lemmatized_text = ' '.join(lemmatized_tokens)

  return lemmatized_text

# Apply lemmatization to each email
df_emails['Lemm_Cleaned_Email'] = df_emails['Cleaned_Email'].apply(lemmatization)

print(df_emails)

In [ ]:
def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

# Remove punctuation
df_emails['Cleaned_Email'] = df_emails['Cleaned_Email'].apply(remove_punctuation)
df_emails['Email_With_Original_Letters'] = df_emails['Email_With_Original_Letters'].apply(remove_punctuation)
df_emails['Lemm_Cleaned_Email'] = df_emails['Lemm_Cleaned_Email'].apply(remove_punctuation)

In [ ]:
# Replace numbers
df_emails['Cleaned_Email'] = df_emails['Cleaned_Email'].str.replace(r'\d+', '', regex = True)
df_emails['Email_With_Original_Letters'] = df_emails['Email_With_Original_Letters'].str.replace(r'\d+', '', regex = True)
df_emails['Lemm_Cleaned_Email'] = df_emails['Lemm_Cleaned_Email'].str.replace(r'\d+', '', regex = True)

In [ ]:
def remove_single_letters(email):

  # Split the email into words
  words = word_tokenize(email)

  return ' '.join([word for word in words if len(word) > 2 ]) #len(word) > 2 and len(word) < 16

# Remove words with length < 2
df_emails['Cleaned_Email'] = df_emails['Cleaned_Email'].apply(remove_single_letters)
df_emails['Email_With_Original_Letters'] = df_emails['Email_With_Original_Letters'].apply(remove_single_letters)
df_emails['Lemm_Cleaned_Email'] = df_emails['Lemm_Cleaned_Email'].apply(remove_single_letters)

In [ ]:
def check_huge_words(email):

  # A boolean value with values 0,1
  # If there is no word bigger than 15 letters, then it is set to 0
  # If there is at least one word bigger than 15 letters, then it is set to 1
  huge_word_exists = 0

  # Split the email into words
  words = word_tokenize(email)

  for word in words:
    if len(word) > 15:
      huge_word_exists = 1
      break

  return huge_word_exists

df_emails['Huge_Word_Exists'] = df_emails['Cleaned_Email'].apply(check_huge_words)

In [ ]:
def remove_huge_words(email):

  # Split the email into words
  words = word_tokenize(email)

  return ' '.join([word for word in words if len(word) < 16]) #15

df_emails['Cleaned_Email'] = df_emails['Cleaned_Email'].apply(remove_huge_words)
df_emails['Email_With_Original_Letters'] = df_emails['Email_With_Original_Letters'].apply(remove_huge_words)
df_emails['Lemm_Cleaned_Email'] = df_emails['Lemm_Cleaned_Email'].apply(remove_huge_words)

In [ ]:
# Tokenize the emails and create a column for them
df_emails['Tokenized_Email'] = df_emails['Cleaned_Email'].apply(word_tokenize)

In [ ]:
def find_duplicate_words(email):
  '''
  Find words that are appear in email more than once.

  Args:
    email (str): The body (text) of each email

  Return:
    The dictionary of duplicate words with keys the
    words and values their frequencies.
  '''

  # Split the email into words
  words_list = word_tokenize(email)

  # Count the occurrences of each word
  word_counts = Counter(words_list)

  # Find words that appear more than once - not punctuation
  duplicate_words = { word:count for word, count in word_counts.items() if count > 1 and word.isalpha()}

  return duplicate_words

# Find if there are duplicate words in each email
df_emails['Duplicates'] = df_emails['Lemm_Cleaned_Email'].apply(find_duplicate_words)

print(df_emails)

In [ ]:
df_emails = df_emails.drop(df_emails[df_emails['Lemm_Cleaned_Email'] == ''].index)
print(df_emails)

# **Co-occurrence Analysis**

**Bigrams**

In [ ]:
def extract_bigrams(text):
    tokens = nltk.word_tokenize(text)
    return list(nltk.bigrams(tokens))

bigram_data = []
for _, row in df_emails.iterrows():
    bigrams_list = extract_bigrams(row['Lemm_Cleaned_Email'])
    bigram_data.extend(bigrams_list)

print(bigram_data)
print('Length of bigrams list: {}'.format(len(bigram_data)))
print('Length of bigrams set: {}'.format(len(set(bigram_data))))

**Trigrams**

In [ ]:
def extract_trigrams(text):
    tokens = nltk.word_tokenize(text)
    return list(nltk.trigrams(tokens))

trigram_data = []
for _, row in df_emails.iterrows():
    trigrams_list = extract_trigrams(row['Lemm_Cleaned_Email'])
    trigram_data.extend(trigrams_list)

print(trigram_data)
print('Length of trigrams list: {}'.format(len(trigram_data)))
print('Length of trigrams set: {}'.format(len(set(trigram_data))))

In [ ]:
# Create a set of bigrams for quick lookup
bigram_set = set(bigram_data)

# Initialize a dictionary to store trigrams associated with each bigram
aggregated_trigrams = defaultdict(list)

# Iterate through each trigram and check if it contains any of the bigrams
for trigram in trigram_data:
    bigram1 = (trigram[0], trigram[1])
    bigram2 = (trigram[1], trigram[2])

    if bigram1 in bigram_set:
        aggregated_trigrams[bigram1].append(trigram)
    if bigram2 in bigram_set:
        aggregated_trigrams[bigram2].append(trigram)

# Filter to keep only those bigrams with at least two associated trigrams
filtered_aggregated_trigrams = {bigram: trigrams for bigram, trigrams in aggregated_trigrams.items() if len(trigrams) >= 2}

# Print the results
for bigram, trigrams in filtered_aggregated_trigrams.items():
    print(f"Bigram: {bigram}, Trigrams: {trigrams}")

# **TF-IDF**

In [ ]:
# Initialize a TF-IDF vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the emails
tfidf_matrix = vectorizer.fit_transform(df_emails['Lemm_Cleaned_Email'])

# Create a dataFrame to display the TF-IDF matrix
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

# tfidf_df now contains the TF-IDF scores for each term in each email
print(tfidf_df)

In [ ]:
cosine_similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Print the cosine similarities
print("Cosine Similarity Matrix:")
print(cosine_similarity_matrix)

In [ ]:
# Summarize the rows of tf-idf dataframe
sum_tfidf_rows= tfidf_df.sum(axis=0)

# Save them in a dataframe and sort them by descending order
important_terms = pd.DataFrame(sum_tfidf_rows, columns=['Important_Terms'])
sorted_important_terms = important_terms.sort_values(by='Important_Terms', ascending=False)

print(sorted_important_terms)

In [ ]:
# Summarize the columns of tf-idf dataframe
sum_tfidf_cols= tfidf_df.sum(axis=1)

# Save them in a dataframe and sort them by descending order
important_emails = pd.DataFrame(sum_tfidf_cols, columns=['Email_Importance'])
sorted_important_emails = important_emails.sort_values(by='Email_Importance', ascending=False)

print(sorted_important_emails)

In [ ]:
# Important terms dictionary based on TF-IDF score
important_terms_dict = sum_tfidf_rows.to_dict()

filtered_bigrams = []
for b in filtered_aggregated_trigrams.keys():

  # Check if both words of bigram are important based on TF-IDF score
  if b[0] in important_terms_dict.keys() and b[1] in important_terms_dict.keys():

    # Check if at least one word has TF-IDF score higher than 80
    if important_terms_dict[b[0]] > 80 or important_terms_dict[b[1]] > 80:
      filtered_bigrams.append((b[0], b[1]))
    else:
      continue

# **Filter Emails**

In [ ]:
def filter_emails(bigrams, emails):

  filtered_emails = pd.DataFrame(columns=['Index', 'Lemm_Cleaned_Email'])  # Initialize an empty DataFrame with column names
  for index, email in emails.iterrows():
      for bigram in bigrams:
          if bigram[0] in email['Lemm_Cleaned_Email'] and bigram[1] in email['Lemm_Cleaned_Email']:
              filtered_emails = pd.concat([filtered_emails, pd.DataFrame([email])], ignore_index=True) # Append the matching email
              break  # No need to continue checking other bigrams if one is found
  return filtered_emails

filtered_emails = filter_emails(filtered_bigrams, df_emails)

In [ ]:
# Spot and keep only one occurrence of each email
filtered_emails.drop_duplicates(subset=['Lemm_Cleaned_Email'], keep='first', inplace=True)
filtered_emails['Index'] = filtered_emails.index

In [ ]:
filtered_emails

# **F I L E 1**

*   **CREATE THE CSV FILE ✅** texts_with_properties.csv



In [ ]:
selected_cols = ['Index', 'Lemm_Cleaned_Email', 'Url_Exists', 'Urls', 'Length',
                 'Special_Characters', 'Per_Email_Uppercase_Percentage',
                 'Numbers_With_Context', 'Huge_Word_Exists', 'Duplicates',
                 'NER_Clean_Results']

filtered_emails_selected = filtered_emails[selected_cols]
filtered_emails_selected.head()

filtered_emails_selected.to_csv('texts_with_properties.csv', index=False)

print('CSV file generated sucessfully!')

# **F I L E  2**

*   **CREATE THE CSV FILE ✅** common_words.csv

In [ ]:
def calculate_common_words(email1, email2):

	'''
	 Calculate the number of common words between two emails

	Args:
		email1: The body/content of the one email.
		email2: The body/content of the another email.

	Returns:
		The number of common words between these two emails.
	'''

	words1 = set(word_tokenize(email1))
	words2 = set(word_tokenize(email2))

	return len(words1.intersection(words2))

# Generate email pairs and common word count
email_pairs = []
for i in filtered_emails['Index'][:100]:
	for j in filtered_emails['Index'][:100]:
		if i < j:
			common = calculate_common_words(filtered_emails['Lemm_Cleaned_Email'][i], filtered_emails['Lemm_Cleaned_Email'][j])
			if common > 9:
				email_pairs.append((i, j, common))

# Write results to CSV file
with open('common_words.csv', 'w', newline='') as csvfile:
	columns = ['email1_id', 'email2_id', 'common_words']
	writer = csv.DictWriter(csvfile, fieldnames = columns)

	writer.writeheader()
	for pair in email_pairs:
		writer.writerow({'email1_id':pair[0], 'email2_id': pair[1], 'common_words': pair[2]})

print('CSV file generated sucessfully!')


# **F I L E 3 & 4**

In [ ]:
# New bigram set
new_bigram_data = []
for _, row in filtered_emails.iterrows():
    new_bigrams_list = extract_bigrams(row['Lemm_Cleaned_Email'])
    new_bigram_data.extend(new_bigrams_list)

# New trigram set
new_trigram_data = []
for _, row in filtered_emails.iterrows():
    new_trigrams_list = extract_trigrams(row['Lemm_Cleaned_Email'])
    new_trigram_data.extend(new_trigrams_list)

print('Trigram data with duplicates:', len(new_trigram_data))

# Create a set of bigrams for quick lookup
new_bigram_set = set(new_bigram_data)

# Remove trigram duplicates
new_trigram_data = list(set(new_trigram_data))
print('Trigram data without duplicates:', len(new_trigram_data))

# Initialize a dictionary to store trigrams associated with each bigram
new_aggregated_trigrams = defaultdict(list)

# Iterate through each trigram and check if it contains any of the bigrams
for trigram in new_trigram_data:
    bigram1 = (trigram[0], trigram[1])
    bigram2 = (trigram[1], trigram[2])

    if bigram1 in new_bigram_set:
        new_aggregated_trigrams[bigram1].append(trigram)
    if bigram2 in new_bigram_set:
        new_aggregated_trigrams[bigram2].append(trigram)

# Filter to keep only those bigrams with at least two associated trigrams
new_filtered_aggregated_trigrams = {bigram: trigrams for bigram, trigrams in new_aggregated_trigrams.items() if len(trigrams) >= 2}

# Print the results
for bigram, trigrams in new_filtered_aggregated_trigrams.items():
    print(f"Bigram: {bigram}, Trigrams: {trigrams}")

*   **CREATE THE CSV FILE ✅** trigram.csv


In [ ]:
# Create a dictionary with the index of the email as key and its trigrams as values
trigrams_with_idx = []

for _, row in filtered_emails.iterrows():
  row_trigrams = extract_trigrams(row['Lemm_Cleaned_Email'])
  for tr in row_trigrams:
    tr_str = ' '.join(tr)
    trigrams_with_idx.append((row['Index'], tr_str))

trigrams_df = pd.DataFrame(trigrams_with_idx, columns=['index', 'trigram'])
print(trigrams_df)
trigrams_df.to_csv('trigram.csv', index=False)
print('CSV file generated sucessfully!')


*   **CREATE THE CSV FILE ✅** trigram_to_bigram.csv

In [ ]:
# Initialize list for bigram edges
trigrams_for_nodes = set()
bigrams_for_edges = []

# Iterate through the dictionary
for bigram, trigrams in new_filtered_aggregated_trigrams.items():
    bigram_str = ' '.join(bigram)
    for trigram in trigrams:
        trigram_str = ' '.join(trigram)
        trigrams_for_nodes.add(trigram_str)
        bigrams_for_edges.append((trigram_str, bigram_str))

# Convert to DataFrames
edges_df = pd.DataFrame(bigrams_for_edges, columns=["trigram", "bigram"])
print(edges_df)

# Save to CSV
edges_df.to_csv("trigram_to_bigram.csv", index=False)
print('CSV file generated sucessfully!')


# **Topic Modeling**

In [ ]:
# List of emails
tokens_list = [tokens for tokens in filtered_emails['Lemm_Cleaned_Email'] ]

# List of tokenized emails
tokens_list = [word_tokenize(tokens) for tokens in tokens_list]

# Create a dictionary with the emails' words
dictionary = corpora.Dictionary(tokens_list)

# Convert to bag-of-words corpus
corpus = [dictionary.doc2bow(text) for text in tokens_list]

# Creating document-term matrix
doc_term_matrix = [dictionary.doc2bow(doc) for doc in tokens_list]

# Define the range of topics to iterate over
min_topics = 2
max_topics = 12
step_size = 1
num_topics_range = range(min_topics, max_topics + 1, step_size)

# Feature names of the
feature_names = vectorizer.get_feature_names_out()

coherence_scores = []
model_num_topics = {}

In [ ]:
# Create a function to compute coherence score for a given number of topics
def compute_coherence_score(model, corpus, dictionary, texts, num_topics, coherence='c_v'):

  """
  Loop through a given number of topics and compute coherence score
  for each algorithm .

  Args:
    model: The topic modeling algorithm.
    corpus: The bag-of-words corpus of documents.
    dictionary: The mapping between normalized words and their integer ids.
    texts: The original texts of the documents.
    num_topics: Number of topics to initialize the topic modeling algorithm.
    coherence: Coherence measure to be used.

  Returns:
    The coherence score achieved.
  """

  if model == 'lda':
    lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics,
                                                passes = 15, random_state = 0)
    # Calculate coherence score
    coherence_model_lda = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence=coherence)
    coherence_score = coherence_model_lda.get_coherence()

  elif  model == 'nmf':
    # Fit NMF model with num_topics
    nmf_model = Nmf(corpus=corpus, id2word=dictionary, num_topics=num_topics,
                    passes = 15, random_state = 0)

    # Calculate coherence score
    coherence_model_nmf = CoherenceModel(model = nmf_model, texts=tokens_list, dictionary=dictionary, coherence=coherence)
    coherence_score = coherence_model_nmf.get_coherence()

  elif model == 'lsa':
    lsa_model = LsiModel(doc_term_matrix, num_topics=num_topics,
                         id2word = dictionary, random_seed =0)
    # Calculate coherence score
    coherence_model_lsa = CoherenceModel(model=lsa_model, texts=texts, dictionary=dictionary, coherence=coherence)
    coherence_score = coherence_model_lsa.get_coherence()

  return coherence_score

In [ ]:
def plot_coherence_scores(model, list_coherence_scores):

  """
  Plot of the coherence score in relation to the number of topics.

  Args:
    list_coherence_scores: List of coherence scores for each number of topic.

  Returns: -
  """

  plt.plot(num_topics_range, list_coherence_scores, color="#06C")
  plt.xlabel("Number of Topics")
  plt.ylabel("Coherence Score")
  plt.title("Coherence Score vs Number of Topics")
  plt.suptitle(model.upper())
  plt.grid(color = "lightgrey", ls = ":")
  plt.gca().spines[['top', 'right']].set_visible(False)
  plt.show()

In [ ]:
# Loop through different models to identify the numbers of topics
for model in ['lda', 'nmf', 'lsa']:
  coherence_scores = []
  for num_topics in num_topics_range:
    coherence_score = compute_coherence_score(model, corpus, dictionary, tokens_list, num_topics)
    coherence_scores.append((num_topics, coherence_score))

  print(model.upper(), 'model')
  # Print coherence scores for different numbers of topics
  for num_topics, score in coherence_scores:
    print("Num Topics: {}, Coherence Score: {}".format(num_topics, score))

  # Select the number of topics with the highest coherence score
  best_num_topics = max(coherence_scores, key=lambda x: x[1])[0]
  model_num_topics[model] = best_num_topics
  print("Best Number of Topics:", best_num_topics)

  # Plot coherence scores
  list_coherence_scores = [coherence_score[1] for coherence_score in coherence_scores]
  plot_coherence_scores(model, list_coherence_scores)
  print()

* **Non-negative Matrix Factorization**

In [ ]:
# Apply NMF to the best number of topics in the data
nmf_model = Nmf(corpus=corpus, id2word=dictionary, num_topics=model_num_topics['nmf'],
                    passes = 15, random_state = 0)

# Extract the top words for each topic
top_words_per_topic = []
num_top_words = 5000  # Define how many top words you want per topic

for topic_id in range(model_num_topics['nmf']):
    top_words = nmf_model.show_topic(topic_id, topn=num_top_words)
    top_words_per_topic.append([word for word, _ in top_words])

# **F I L E  5**

*   **CREATE THE CSV FILE ✅** keyword_to_topic.csv

In [ ]:
N_TOP_KEYWORDS = 5000

# Extract the keywords and their probabilities for each topic
words = np.array(vectorizer.get_feature_names_out())

topic_keywords = []
for topic_id in range(nmf_model.num_topics):

  # Get the words and their probabilities for the topic
  topic = nmf_model.get_topic_terms(topic_id, topn = N_TOP_KEYWORDS)

  total_prob = sum(prob for _, prob in topic)

  for word_id, prob in topic:
    word = dictionary[word_id]
    normalized_prob = prob / total_prob
    topic_keywords.append([topic_id, word, normalized_prob])

df = pd.DataFrame(topic_keywords, columns=['Topic', 'Keyword', 'Weight'])
df.to_csv("keyword_to_topic.csv", index=False)

print("CSV file created successfully!")

# **F I L E 6**

*   **CREATE THE CSV FILE ✅** bigram_to_keyword.csv

In [ ]:
# Prepare the CSV data
keyword_in_bigram = []
bigram_for_nodes = list(new_filtered_aggregated_trigrams.keys())
all_keywords = [keywords for topic in top_words_per_topic for keywords in topic]

# Remove duplicate keywords - each keyword is a single nodes
all_keywords = list(set(all_keywords))

for bigram in bigram_for_nodes:
  bigram_str = ' '.join(bigram)
  for keyword in all_keywords:
      if set(keyword.split()) & set(bigram_str.split()):
          keyword_in_bigram.append({"keyword": keyword, "bigram": bigram_str})

# Write to CSV
with open('bigram_to_keyword.csv', 'w', newline='') as csvfile:
    fieldnames = ['keyword', 'bigram']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for row in keyword_in_bigram:
        writer.writerow(row)

print("CSV file created successfully.")